In [1]:
pip install pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 5.2 MB/s eta 0:00:00ta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("Kafka WikiMedia Cleaning").getOrCreate()
spark

In [4]:
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers","kafka1:9092,kafka1:9093,kafka1:9094").option("subscribe", "wikimedia.latestchanges").option("startingOffsets", "earliest").load()

In [5]:
from pyspark.sql.functions import col

In [6]:
df_string = df.selectExpr("CAST(value AS STRING)")

In [7]:
from pyspark.sql.types import *

schema = StructType([
    StructField("$schema", StringType()),

    StructField("meta", StructType([
        StructField("uri", StringType()),
        StructField("request_id", StringType()),
        StructField("id", StringType()),
        StructField("domain", StringType()),
        StructField("stream", StringType()),
        StructField("dt", StringType()),
        StructField("topic", StringType()),
        StructField("partition", IntegerType()),
        StructField("offset", LongType())
    ])),

    StructField("id", LongType()),
    StructField("type", StringType()),
    StructField("namespace", IntegerType()),
    StructField("title", StringType()),
    StructField("title_url", StringType()),
    StructField("comment", StringType()),
    StructField("timestamp", LongType()),
    StructField("user", StringType()),
    StructField("bot", BooleanType()),

    StructField("log_id", LongType()),
    StructField("log_type", StringType()),
    StructField("log_action", StringType()),
    StructField("log_action_comment", StringType()),

    StructField("log_params", StructType([
        StructField("duration", StringType()),
        StructField("flags", StringType()),
        StructField("sitewide", BooleanType()),
        StructField("blockId", LongType())
    ])),

    StructField("notify_url", StringType()),
    StructField("minor", BooleanType()),
    StructField("patrolled", BooleanType()),

    StructField("length", StructType([
        StructField("old", LongType()),
        StructField("new", LongType())
    ])),

    StructField("revision", StructType([
        StructField("old", LongType()),
        StructField("new", LongType())
    ])),

    StructField("server_url", StringType()),
    StructField("server_name", StringType()),
    StructField("server_script_path", StringType()),
    StructField("wiki", StringType()),
    StructField("parsedcomment", StringType())
])

In [8]:
from pyspark.sql.functions import from_json, col

df_parsed = df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

In [9]:
df_flat = df_parsed.select(
    "type",
    "user",
    "title",
    "timestamp",
    "wiki",
    col("meta.domain").alias("domain"),
    col("meta.stream").alias("stream"),
    col("revision.new").alias("rev_new"),
    col("revision.old").alias("rev_old")
)

In [10]:
df_flat.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "/home/jovyan/work/output/final") \
    .option("checkpointLocation", "/home/jovyan/work/checkpoint/output/final") \
    .start() \
    .awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = db0d116e-f657-411f-b377-a6d14c951dee, runId = fbecc790-e03b-4225-9e89-c788de1232ec] terminated with exception: Partition wikimedia.latestchanges-0's offset was changed from 46645 to 2215, some data may have been missed. 
Some data may have been lost because they are not available in Kafka any more; either the
 data was aged out by Kafka or the topic may have been deleted before all the data in the
 topic was processed. If you don't want your streaming query to fail on such cases, set the
 source option "failOnDataLoss" to "false".
    